# HOMEWORK 10

Done by: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/ComputerVision-PBY_HW10

In this homework, you are going to use and compare two different trackers (of your liking) and compare the results.

In [ ]:
# # Silent installation or update
#
# import sys
# import apps.main as mn
#
# # Clean cache
# !"{sys.executable}" -m pip cache purge -q
# !"{sys.executable}" -m pip uninstall -y opencv-python opencv-python-headless -q
#
# # Force updating
# package_update = [
#     "pip",
#     "scikit-learn",
#     "pandas",
# ]
#
# for package_name in package_update:
#     !"{sys.executable}" -m pip install -U "{package_name}" -q
#
# # Install missing packages
# package_array = [
#     "jinja2",
#     "ipywidgets",
#     "nbformat",
#     "numpy",
#     "matplotlib",
#     "opencv-contrib-python<5",
#     "moviepy",
# ]
#
# for package_name in package_array:
#     if not mn.is_package_satisfied(package_name):
#         !"{sys.executable}" -m pip install "{package_name}" -q
#
# !"{sys.executable}" -m pip show "ultralytics" > /dev/null 2>&1 || !"{sys.executable}" -m pip install "ultralytics" --no-deps -q

In [ ]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input videos
# hm_version = 10
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/ComputerVision-PBY_HW{hm_version}.git"
# main_file_name = f"Bohdan_Pinchuk_CV_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone videos
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name, ".git", ".idea"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"
#
# # Note. To upload big files to GitHub, execute the next command in the terminal
# # 1. brew install git-lfs
# # 2. git lfs install
# # 3.1. git lfs track "*.safetensors"
# # 3.2. git lfs track "*.bin"

### Step 1

Decide what video you are going to use for this homework, select an object and generate the template. You can use any video you want (your own, from Youtube, etc.)
and track any object you want (e.g. a car, a pedestrian, etc.).

In [ ]:
# Load data

import cv2
import apps.reporter as rpt
from pathlib import Path
from moviepy import VideoFileClip
from IPython.display import Video

# Input data
file_name = "resources/videos/Doha_Qatar_Dashcam_Drive.mp4"

# Solution
file_path_input = Path(file_name)
file_path_output = file_path_input.with_suffix(".webm")

# Load data
if not file_path_output.exists():
    if file_path_input.exists():
        # convert video in necessary format
        clip = VideoFileClip(file_path_input)
        clip.write_videofile(file_path_output, codec="libvpx")
        clip.close()
    else:
        raise FileNotFoundError("File doesn't exist!")

# Analysis video
video = cv2.VideoCapture(file_path_output)

# check if the video can open
if video.isOpened():
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
else:
    raise ValueError('Incorrect input data "video"!')

# memory release
# video.release()

rp = rpt.Reporter()
rp.tolerance = 2
rp.add_item("Source of the video", rp.format_link("https://www.youtube.com/watch?v=xX31l9Us1TA"))
rp.add_item("(width, height)", str((width, height)))
rp.add_item("FPS", rp.format_value(fps))
rp.add_item("Total frames", str(total_frames))
rp.add_item("Duration, seconds", rp.format_value(duration))

# Print results
rp.print_pd_report("Video metadata")

Video(file_path_output, embed=False, html_attributes="controls width=720")


In [ ]:
# Detection

import cv2
import itertools
import numpy as np
import apps.reporter as rpt
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Input data
car_id = 0
classes = [2]  # for car type is 2
conf = 0.4
thickness = 5
color_palette_name = 'tab20'
car_init_box_xyxy = None
car_init_box_xywh = None
car_init_color = None

# Solution
# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

ok, first_frame = video.read()
assert ok

# get colors map
cmap = (np.asarray(plt.get_cmap(color_palette_name).colors) * 255).astype(int).tolist()  # type: ignore
# extract colors and convert
colors = itertools.cycle(tuple(color) for color in cmap)

# Convert it to RGB
first_frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
first_frame_dt = first_frame_rgb.copy()

model = YOLO('resources/models/yolov8s.pt')
results = model(first_frame_rgb, classes=classes, conf=conf, verbose=False)
# get car coordinates
car_boxes_xyxy = results[0].boxes.xyxy.cpu().numpy().astype(int)

# draw rectangle over the cars
for idx, car_box_xy in enumerate(car_boxes_xyxy):
    # extract coordinates
    x1, y1, x2, y2 = car_box_xy
    # get color
    current_color = next(colors)
    # draw rectangle
    cv2.rectangle(first_frame_dt, (x1, y1), (x2, y2), current_color, thickness)
    # add text
    cv2.putText(first_frame_dt, f"Car ID: {idx}", (x1, y1 - 5), 0, 1, current_color, 2)
    # save coordinates
    if idx == car_id:
        car_init_box_xyxy = car_box_xy
        car_init_box_xywh = np.asarray([x1, y1, x2 - x1, y2 - y1])
        car_init_color = current_color

rp = rpt.Reporter()
rp.add_item("Car ID", str(car_id))
rp.add_item("Car coordinates (x1, y1, x2, y2)", str(car_init_box_xyxy))
rp.add_item("Car coordinates (x, y, w, h)", str(car_init_box_xywh))

# Print results
rp.print_pd_report("Bounding Box")

images_dict = {
    "First frame": first_frame_rgb,
    "Detected cars": first_frame_dt,
}


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = images_dict
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 5 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Input data")
plt.tight_layout()
plt.show()


### Step 2

Initialize a tracker (e.g. KCF).

### Step 3

Run the tracker on the video and the selected object. Run the tracker for around 10-15 frames.

### Step 4

For each frame, print the bounding box on the image and save it.

In [ ]:
# Tracker KCF

import cv2
import numpy as np

# Input data
n_frames = 15
step_frames = 10

# Solution
frame_bins = np.linspace(0, total_frames - 1, n_frames + 1, dtype=int)

tracker = cv2.TrackerKCF.create()  # type:ignore
tracker.init(first_frame, car_init_box_xywh)

# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

tracked_images = {}

for idx in range(total_frames):
    # read frame
    ok, frame = video.read()
    if not ok:
        break

    # optimization
    if idx % step_frames == 0 or idx in frame_bins:
        # update box
        ok, box = tracker.update(frame)
        if ok and idx in frame_bins:
            x, y, w, h = box
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # draw rectangle
            if car_init_color is not None:
                cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), car_init_color, thickness)

            # save image
            tracked_images[f"frame: {idx}"] = frame_rgb

# Print results


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = tracked_images
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 4 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Tracker KCF", y=0.99)
plt.tight_layout()
plt.show()


### Step 5

Select a different tracker (e.g. CSRT) and repeat steps 2, 3 and 4.

In [ ]:
# Tracker CSRT

import cv2
import numpy as np

# Input data
n_frames = 15
step_frames = 10

# Solution
frame_bins = np.linspace(0, total_frames - 1, n_frames + 1, dtype=int)

tracker = cv2.TrackerCSRT.create()  # type:ignore
tracker.init(first_frame_rgb, car_init_box_xywh)

# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

tracked_images = {}

for idx in range(total_frames):
    # read frame
    ok, frame = video.read()
    if not ok:
        break

    # optimization
    if idx % step_frames == 0 or idx in frame_bins:
        # update box
        ok, box = tracker.update(frame)
        if ok and idx in frame_bins:
            x, y, w, h = box
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # draw rectangle
            if car_init_color is not None:
                cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), car_init_color, thickness)

            # save image
            tracked_images[f"frame: {idx}"] = frame_rgb

# Print results


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = tracked_images
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 4 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Tracker CSRT", y=0.99)
plt.tight_layout()
plt.show()


In [ ]:
# Tracker MIL

import cv2
import numpy as np

# Input data
n_frames = 15
step_frames = 10

# Solution
frame_bins = np.linspace(0, total_frames - 1, n_frames + 1, dtype=int)

tracker = cv2.TrackerMIL_create()  # type:ignore
tracker.init(first_frame, car_init_box_xywh)

# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

tracked_images = {}

for idx in range(total_frames):
    # read frame
    ok, frame = video.read()
    if not ok:
        break

    # optimization
    if idx % step_frames == 0 or idx in frame_bins:
        # update box
        ok, box = tracker.update(frame)
        if ok and idx in frame_bins:
            x, y, w, h = box
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # draw rectangle
            if car_init_color is not None:
                cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), car_init_color, thickness)

            # save image
            tracked_images[f"frame: {idx}"] = frame_rgb

# Print results


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = tracked_images
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 4 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Tracker MIL", y=0.99)
plt.tight_layout()
plt.show()


In [ ]:
# Tracker VitTrack

import cv2
import numpy as np

# Input data
n_frames = 15
step_frames = 10
model_path = "resources/models/object_tracking_vittrack_2023sep.onnx"

# Solution
frame_bins = np.linspace(0, total_frames - 1, n_frames + 1, dtype=int)

params = cv2.TrackerVit_Params()  # type:ignore
params.net = model_path
tracker = cv2.TrackerVit.create(params)  # type:ignore
tracker.init(first_frame, car_init_box_xywh)  # type:ignore

# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

tracked_images = {}

for idx in range(total_frames):
    # read frame
    ok, frame = video.read()
    if not ok:
        break

    # optimization
    if idx % step_frames == 0 or idx in frame_bins:
        # update box
        ok, box = tracker.update(frame)
        if ok and idx in frame_bins:
            x, y, w, h = box
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # draw rectangle
            if car_init_color is not None:
                cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), car_init_color, thickness)

            # save image
            tracked_images[f"frame: {idx}"] = frame_rgb

# Print results


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = tracked_images
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 4 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Tracker VitTrack", y=0.99)
plt.tight_layout()
plt.show()


In [ ]:
# Tracker NanoTrack

import cv2
import numpy as np

# Input data
n_frames = 15
step_frames = 10
model_path_backbone = "resources/models/nanotrack_backbone_sim.onnx"
model_path_neckhead = "resources/models/nanotrack_head_sim.onnx"

# Solution
frame_bins = np.linspace(0, total_frames - 1, n_frames + 1, dtype=int)

params = cv2.TrackerNano_Params()  # type:ignore
params.backbone = model_path_backbone
params.neckhead = model_path_neckhead
tracker = cv2.TrackerNano.create(params)  # type:ignore
tracker.init(first_frame, car_init_box_xywh)  # type:ignore

# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

tracked_images = {}

for idx in range(total_frames):
    # read frame
    ok, frame = video.read()
    if not ok:
        break

    # optimization
    if idx % step_frames == 0 or idx in frame_bins:
        # update box
        ok, box = tracker.update(frame)
        if ok and idx in frame_bins:
            x, y, w, h = box
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # draw rectangle
            if car_init_color is not None:
                cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), car_init_color, thickness)

            # save image
            tracked_images[f"frame: {idx}"] = frame_rgb

# Print results


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = tracked_images
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 4 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Tracker NanoTrack", y=0.99)
plt.tight_layout()
plt.show()


In [ ]:
# Tracker DaSiamRPN

import cv2
import numpy as np

# Input data
n_frames = 15
step_frames = 10
model_path = "resources/models/dasiamrpn_model.onnx"
model_path_kernel_r1 = "resources/models/dasiamrpn_kernel_r1.onnx"
model_path_kernel_cls1 = "resources/models/dasiamrpn_kernel_cls1.onnx"

# Solution
frame_bins = np.linspace(0, total_frames - 1, n_frames + 1, dtype=int)

params = cv2.TrackerDaSiamRPN_Params()  # type:ignore
params.model = model_path
params.kernel_r1 = model_path_kernel_r1
params.kernel_cls1 = model_path_kernel_cls1
params.backend = cv2.dnn.DNN_BACKEND_OPENCV
params.target = cv2.dnn.DNN_TARGET_CPU
tracker = cv2.TrackerDaSiamRPN.create(params)  # type:ignore
tracker.init(first_frame, car_init_box_xywh)  # type:ignore

# reset video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

tracked_images = {}

for idx in range(total_frames):
    # read frame
    ok, frame = video.read()
    if not ok:
        break

    # optimization
    if idx % step_frames == 0 or idx in frame_bins:
        # update box
        ok, box = tracker.update(frame)
        if ok and idx in frame_bins:
            x, y, w, h = box
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # draw rectangle
            if car_init_color is not None:
                cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), car_init_color, thickness)

            # save image
            tracked_images[f"frame: {idx}"] = frame_rgb

# Print results


In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_map = tracked_images
n_img_rows = math.ceil(len(images_map) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 4 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_map.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_map), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Tracker DaSiamRPN", y=0.99)
plt.tight_layout()
plt.show()


### Step 6

Compare the results:
* Do you see any differences? If so, what are they?
* Does one tracker perform better than the other? In what way?

### Conclusion
1. Target Loss in **KCF** and **CSRT**: Both **KCF** and **CSRT** produced only 6 out of 16 frames. Around frame 1749, as the car moved and changed scale, both algorithms lost the target (`ok = False`) and stopped tracking.
2. False Tracking in **MIL**: Although **MIL** generated all 16 frames, it suffered from severe drift. By frame 5, the bounding box lost the car and got stuck on the road surface, continuing to report success silently.
3. Worst Results with Neural Trackers: In this case, deep learning trackers (**NanoTrack**, **VitTrack**, and **DaSiamRPN**) showed the worst performance. Despite outputting all 16 frames, they completely lost the car right after the first frame, and their bounding boxes expanded enormously over the empty road and background.
4. Frame Skipping Optimization: Using `step_frames = 10` reduced computational load by ~90% across the 5,250 frames without degrading the keyframe visual results.